<a href="https://colab.research.google.com/github/srushtiganesh/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 23.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=cb9448395ad50092f783ab86cd8e0f211b2ceaabbf1e4e3b243a97817cd101a8
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


# Implement a demonstration of the BB84 protocol, with and without an at-
tacker. Although BB84 is a communication protocol involving two separate
agents (or three, including an attacker), implement it as a single sequential
program — but make it clear which parts of the code correspond to each
agent. To generate the random choices that are needed by the protocol,
measure a suitable quantum state such as 1√2 (|0⟩ + |1⟩). Don’t use a stan-
dard Python random number generator. The attacker should try to obtain
information about the key, somehow, for example by measuring some of the
qubits. You can also try other kinds of attack. The participants in the
protocol should report an attack by using some threshold on the proportion
of disrupted qubits

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [5]:
# ============================================================
#  BB84 Quantum Key Distribution — NO ATTACKER
#
#  Colab setup (run first):
#    !pip install qiskit==1.2.4
#    %pip install qiskit-aer==0.15.1
#    %pip install pylatexenc==2.10
#
#  Rules satisfied:
#    - ALL randomness comes from measuring |+> = H|0>
#    - No Python random / numpy random anywhere
#    - n bits generated in ONE batched quantum job (n-qubit circuit)
#    - Agent blocks are clearly labelled
# ============================================================

from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex, plot_histogram
from qiskit.quantum_info import Operator, Statevector
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit import ControlledGate
import math

# ─────────────────────────────────────────────────────────────
#  IMPORTS  (paste these into your Colab cell as-is)
# ─────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit, transpile
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex, plot_histogram
from qiskit.quantum_info import Operator, Statevector
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.circuit import ControlledGate

simulator = BasicSimulator()

# ─────────────────────────────────────────────────────────────
#  QUANTUM RANDOM NUMBER GENERATOR
#
#  Builds a single n-qubit circuit where every qubit is
#  prepared in |+> = H|0> = 1/sqrt(2)(|0> + |1>) and then
#  measured.  One job, one shot — all n bits are independent
#  because each qubit is in its own superposition.
#
#  This satisfies "measure a suitable quantum state" without
#  using Python's random module at all.
# ─────────────────────────────────────────────────────────────
def quantum_random_bits(n: int) -> list:
    """
    Generate n independent quantum-random bits.
    BasicSimulator supports at most 24 qubits, so we batch
    into chunks of 24, each chunk a single job with one shot.
    Each qubit is prepared as H|0> = 1/sqrt(2)(|0>+|1>).
    No standard RNG is used anywhere.
    Returns a list of n ints, each 0 or 1.
    """
    MAX_QUBITS = 24
    bits = []
    remaining = n
    while remaining > 0:
        chunk = min(remaining, MAX_QUBITS)
        qc = QuantumCircuit(chunk, chunk)
        for i in range(chunk):
            qc.h(i)              # |0> -> 1/sqrt(2)(|0> + |1>)
            qc.measure(i, i)
        job = simulator.run(transpile(qc, simulator), shots=1)
        # Qiskit bitstring is q[chunk-1]...q[0], so reverse it
        bitstring = list(job.result().get_counts().keys())[0]
        bitstring = bitstring.replace(" ", "")
        bits.extend(int(b) for b in reversed(bitstring))
        remaining -= chunk
    return bits[:n]


# ─────────────────────────────────────────────────────────────
#  BASIS ENCODING / DECODING
#
#  Basis 0 = rectilinear {|0>, |1>}   labelled (+)
#  Basis 1 = diagonal    {|+>, |->}   labelled (x)
#
#  Encoding table:
#    bit=0, basis=0  ->  |0>
#    bit=1, basis=0  ->  |1>
#    bit=0, basis=1  ->  |+> = H|0>
#    bit=1, basis=1  ->  |-> = HX|0>
# ─────────────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════╗
# ║  AGENT: ALICE                                            ║
# ╚══════════════════════════════════════════════════════════╝
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """
    Alice encodes one classical bit into a single-qubit state.
    The returned QuantumCircuit represents the qubit she sends.
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)      # |0> -> |1>
    if basis == 1:
        qc.h(0)      # rotate into diagonal basis
    return qc


# ╔══════════════════════════════════════════════════════════╗
# ║  AGENT: BOB                                              ║
# ╚══════════════════════════════════════════════════════════╝
def bob_measure(qc: QuantumCircuit, basis: int) -> int:
    """
    Bob measures a received qubit in his chosen basis.
    If basis=1 he first rotates back from the diagonal basis,
    then measures. Returns 0 or 1.
    """
    qc = qc.copy()
    if basis == 1:
        qc.h(0)      # undo diagonal rotation before measuring
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    counts = job.result().get_counts()
    return int(list(counts.keys())[0])


# ╔══════════════════════════════════════════════════════════╗
# ║  BB84 PROTOCOL — NO ATTACKER                             ║
# ╚══════════════════════════════════════════════════════════╝
def run_bb84_no_attacker(n_bits: int = 100, error_threshold: float = 0.15):
    """
    Full BB84 key exchange between Alice and Bob with no Eve.

    Protocol steps
    --------------
    1. ALICE   draws random bits + bases (quantum), encodes qubits
    2. BOB     draws random bases (quantum), measures each qubit
    3. PUBLIC  Alice and Bob compare bases over classical channel
    4. SIFT    discard positions where bases differ
    5. CHECK   sacrifice ~20% of sifted key to estimate QBER
    6. REPORT  declare secure if QBER <= threshold, else abort
    """
    print("=" * 58)
    print("  BB84 — NO ATTACKER")
    print("=" * 58)

    # ----------------------------------------------------------
    # ALICE: generate random bits and bases, encode qubits
    # ----------------------------------------------------------
    print(f"\n[ALICE] Generating {n_bits} random bits and bases ...")
    alice_bits  = quantum_random_bits(n_bits)   # the secret bits
    alice_bases = quantum_random_bits(n_bits)   # 0=rectilinear, 1=diagonal

    # Alice encodes each bit into a qubit and "sends" it to Bob.
    # In hardware this is a photon over a fibre; here we pass
    # the QuantumCircuit object directly (sequential simulation).
    qubits_sent = [alice_encode(bit, basis)
                   for bit, basis in zip(alice_bits, alice_bases)]

    print(f"  bits  (first 10): {alice_bits[:10]}")
    print(f"  bases (first 10): {alice_bases[:10]}   (0=+, 1=x)")

    # ----------------------------------------------------------
    # BOB: choose random measurement bases, measure each qubit
    # ----------------------------------------------------------
    print(f"\n[BOB]   Choosing {n_bits} random bases and measuring ...")
    bob_bases   = quantum_random_bits(n_bits)
    bob_results = [bob_measure(qc, basis)
                   for qc, basis in zip(qubits_sent, bob_bases)]

    print(f"  bases   (first 10): {bob_bases[:10]}")
    print(f"  results (first 10): {bob_results[:10]}")

    # ----------------------------------------------------------
    # PUBLIC CHANNEL: basis reconciliation
    # Alice and Bob broadcast their basis choices (not the bits).
    # They keep only positions where both chose the same basis.
    # ----------------------------------------------------------
    print(f"\n[PUBLIC] Alice and Bob compare bases ...")
    matching_idx = [i for i in range(n_bits)
                    if alice_bases[i] == bob_bases[i]]
    print(f"  Matching positions: {len(matching_idx)} / {n_bits}")

    # ----------------------------------------------------------
    # SIFT: extract the shared key from matching positions
    # ----------------------------------------------------------
    alice_sifted = [alice_bits[i]   for i in matching_idx]
    bob_sifted   = [bob_results[i]  for i in matching_idx]
    sifted_len   = len(alice_sifted)
    print(f"\n[SIFT]  Sifted key length: {sifted_len} bits")

    # ----------------------------------------------------------
    # CHECK: sacrifice first ~20% of sifted key to estimate QBER
    # In a noise-free, attack-free channel QBER should be ~0%.
    # ----------------------------------------------------------
    check_n    = max(1, sifted_len // 5)
    errors     = sum(a != b for a, b in
                     zip(alice_sifted[:check_n], bob_sifted[:check_n]))
    qber       = errors / check_n

    # Remaining bits (after sacrificed check bits) form the final key
    final_alice = alice_sifted[check_n:]
    final_bob   = bob_sifted[check_n:]
    key_len     = len(final_alice)

    print(f"\n[CHECK] Sacrificing {check_n} bits to estimate error rate ...")
    print(f"  Errors : {errors} / {check_n}")
    print(f"  QBER   : {qber:.2%}")
    print(f"  Threshold: {error_threshold:.0%}")

    # ----------------------------------------------------------
    # REPORT: declare outcome
    # ----------------------------------------------------------
    attack_detected = qber > error_threshold
    print("\n[RESULT]")
    if attack_detected:
        print(f"  WARNING: ATTACK DETECTED (QBER {qber:.2%} > {error_threshold:.0%})")
        print("  Alice and Bob abort the key exchange.")
    else:
        keys_match = all(a == b for a, b in zip(final_alice, final_bob))
        print(f"  OK: Channel is secure (QBER {qber:.2%} <= {error_threshold:.0%})")
        print(f"  OK: Final key length : {key_len} bits")
        print(f"  OK: Keys match       : {keys_match}")

    return {
        "qber"            : qber,
        "sifted_len"      : sifted_len,
        "key_len"         : key_len,
        "attack_detected" : attack_detected,
    }


# ─────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────
N         = 100    # number of qubits exchanged
THRESHOLD = 0.15   # QBER threshold for attack detection (15%)

result = run_bb84_no_attacker(n_bits=N, error_threshold=THRESHOLD)

print("\n" + "=" * 58)
print("  SUMMARY")
print("=" * 58)
print(f"  QBER             : {result['qber']:.2%}")
print(f"  Sifted key length: {result['sifted_len']} bits")
print(f"  Final key length : {result['key_len']} bits")
print(f"  Attack detected  : {result['attack_detected']}")
print("=" * 58)

  BB84 — NO ATTACKER

[ALICE] Generating 100 random bits and bases ...
  bits  (first 10): [1, 1, 0, 1, 1, 1, 0, 1, 0, 0]
  bases (first 10): [1, 0, 0, 1, 0, 1, 1, 1, 0, 1]   (0=+, 1=x)

[BOB]   Choosing 100 random bases and measuring ...
  bases   (first 10): [1, 1, 0, 1, 0, 0, 0, 1, 1, 0]
  results (first 10): [1, 1, 0, 1, 1, 0, 1, 1, 0, 1]

[PUBLIC] Alice and Bob compare bases ...
  Matching positions: 47 / 100

[SIFT]  Sifted key length: 47 bits

[CHECK] Sacrificing 9 bits to estimate error rate ...
  Errors : 0 / 9
  QBER   : 0.00%
  Threshold: 15%

[RESULT]
  OK: Channel is secure (QBER 0.00% <= 15%)
  OK: Final key length : 38 bits
  OK: Keys match       : True

  SUMMARY
  QBER             : 0.00%
  Sifted key length: 47 bits
  Final key length : 38 bits
  Attack detected  : False
